# Contextual Compression & Cross-Encoder Relevance Filtering

## 1. The Context Stuffing Bottleneck
Retriever models are optimized for recall over precision. When fetching top-$k$ passages (e.g. $k=10$), up to **80% of retrieved tokens** are boilerplate, introductory fluff, or tangential paragraphs. Dumping all 10 chunks into the LLM prompt:
1. Wastes expensive context window tokens.
2. Increases input token inference cost and latency.
3. Triggers attention dilution, causing the LLM to overlook the exact critical sentence.

### Contextual Compression Pipeline
Contextual Compression dynamically extracts **only the relevant sentences** from retrieved passages:

$$\text{Retrieved Passages } \{P_1, \dots, P_k\} \xrightarrow{\text{Sentence Split}} \{s_1, \dots, s_m\} \xrightarrow{\text{Cross-Encoder}} s(q, s_i) \xrightarrow{\text{Threshold } \tau} \text{Clean Context}$$

1. **Sentence Boundary Splitting**: Split retrieved documents into atomic sentences.
2. **Cross-Encoder Scoring**: Joint attention over $[\text{CLS}] \, q \, [\text{SEP}] \, s_i \, [\text{SEP}]$ captures fine-grained token-level cross-attention impossible with bi-encoders.
3. **Threshold Filtering**: Keep only sentences where relevance score $s(q, s_i) > \tau$.

In [ ]:
import numpy as np

class ContextualCompressor:
    """
    Compresses retrieved text passages by filtering sentences using cross-encoder relevance scoring.
    """
    def __init__(self, threshold=0.5):
        self.threshold = threshold

    def score_sentence(self, query, sentence):
        """
        Simulates a cross-encoder relevance function.
        Returns a score in [0, 1] based on semantic term overlap and key entity matching.
        """
        q_words = set(query.lower().split())
        s_words = set(sentence.lower().split())
        overlap = len(q_words.intersection(s_words))
        
        # Base score from overlap
        score = overlap / max(len(q_words), 1)
        # Bonus for exact key phrase matches
        if any(term in sentence.lower() for term in ["learning rate", "gradient", "convergence"]):
            score = min(1.0, score + 0.3)
        return float(score)

    def compress(self, query, passages):
        compressed_output = []
        total_orig_tokens = 0
        total_comp_tokens = 0
        
        for p_idx, text in enumerate(passages):
            sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 5]
            kept_sentences = []
            
            for s in sentences:
                total_orig_tokens += len(s.split())
                score = self.score_sentence(query, s)
                if score >= self.threshold:
                    kept_sentences.append((score, s))
                    total_comp_tokens += len(s.split())
                    
            if kept_sentences:
                compressed_output.append({
                    "passage_idx": p_idx,
                    "sentences": [s[1] for s in kept_sentences],
                    "avg_score": np.mean([s[0] for s in kept_sentences])
                })
                
        compression_ratio = (1.0 - (total_comp_tokens / max(total_orig_tokens, 1))) * 100
        return compressed_output, compression_ratio

# Verification
raw_passages = [
    "Today's weather is sunny and great for outdoor walks. Deep learning models require an appropriate learning rate for stable convergence. Without gradient clipping, gradients can explode.",
    "In financial markets, stocks fluctuate based on macroeconomic earnings. AdamW optimizer decouples weight decay from learning rate updates to prevent degradation.",
    "The history of aviation began with the Wright brothers in 1903. Aircraft engines operate on thermodynamic Brayton cycles."
]

compressor = ContextualCompressor(threshold=0.35)
user_query = "What prevents gradient explosion and controls learning rate?"
filtered_res, savings = compressor.compress(user_query, raw_passages)

print(f"Query: '{user_query}'")
print(f"Token Space Compressed: {savings:.1f}% reduction in context fluff!
")
for item in filtered_res:
    print(f"Passage #{item['passage_idx']} (Relevance: {item['avg_score']:.2f}):")
    for s in item['sentences']:
        print(f"  + "{s}"")

## 2. Bi-Encoder vs Cross-Encoder Computational Latency Trade-off
- **Bi-Encoder (Vector Search)**: Computes independent embeddings $\mathbf{e}_q$ and $\mathbf{e}_d$. Latency: $O(1)$ lookup via HNSW index. Fast, but lacks cross-token interactions.
- **Cross-Encoder (Re-ranker)**: Concatenates $q$ and $d$ into full transformer self-attention. Latency: $O(K \times (L_q + L_d)^2)$. Slow, but achieves significantly higher ranking precision.

## 3. High-Yield Interview Questions & Best Practices

### Q1: What is the optimal pipeline order when combining Vector Search, Cross-Encoder Re-ranking, and Contextual Compression?
**Answer:** The standard industry funnel is:
1. **Stage 1 (Bi-Encoder Retrieval)**: Fetch top-$k_1$ ($k_1=50$ to 100) candidates using vector cosine similarity or hybrid BM25 + dense search in <15ms.
2. **Stage 2 (Cross-Encoder Re-ranking)**: Score all 100 candidates with a lightweight cross-encoder (e.g. BGE-Reranker-Large or Cohere Rerank) and select top-$k_2$ ($k_2=5$ to 10).
3. **Stage 3 (Contextual Compression)**: Split the top-5 documents into sentences and prune sentences below relevance threshold $\tau$, passing only the filtered tokens to the final generator LLM.

### Q2: How does contextual compression prevent LLM hallucinations?
**Answer:** LLMs hallucinate when provided with ambiguous, self-contradicting, or excessively noisy retrieved context where the model generates a statistically plausible completion rather than grounding strictly in evidence. By removing irrelevant paragraphs, the signal-to-noise ratio in the prompt increases dramatically, allowing direct extraction of the answer.